In [ ]:
# Step 1: Now, we are looking at non-diala users!

import pandas as pd
import numpy as np

# Load Non-DiaLA sheet 
df_nd = pd.read_excel(
    "../mock-data/Active_Inactive_Non-Diala_Mock.xlsx",
    sheet_name="Non_DiaLA_Users"
)
print(f"Raw shape: {df_nd.shape}")

# Drop completely empty rows; similar processes as before!
df_nd = df_nd.dropna(how="all")
print(f"Shape after dropping empty rows: {df_nd.shape}")

# Rename columns to standardised names 
# NOTE: Non-DiaLA has no BP columns, no AOS, no Diab_Duration,
#       no Patient_Status. Follow-up date column is FU_DATE.
df_nd = df_nd.rename(columns={
    "Baseline_Date":              "BL_Entry",
    "FU_DATE":                    "FU_Entry",
    "HDL CHOLESTEROL_BL":         "HDL_BL",
    "SERUM TRIGLYCERIDES_BL":     "TGL_BL",
    "HDL CHOLESTEROL_FU":         "HDL_FU",
    "SERUM TRIGLYCERIDES_FU":     "TGL_FU",
    # Already clean names — listed for clarity:
    # BMI_BL, Waist_BL, HbA1c_BL, Serum Cholesterol_BL
    # BMI_FU, Waist_FU, HbA1c_FU, Serum Cholesterol_FU
})
print(f"\nColumns after rename: {list(df_nd.columns)}")

# Replace 0 with NaN for lab columns (0 not clinically valid) 
zero_cols = [
    "TGL_BL", "TGL_FU",
    "Serum Cholesterol_BL", "Serum Cholesterol_FU",
    "HDL_BL", "HDL_FU"
]
for col in zero_cols:
    if col in df_nd.columns:
        df_nd[col] = df_nd[col].replace(0, np.nan)

# Coerce numeric columns 
numeric_cols = [
    "AGE",
    "HbA1c_BL",  "BMI_BL",  "Waist_BL",
    "HDL_BL",    "TGL_BL",  "Serum Cholesterol_BL",
    "HbA1c_FU",  "BMI_FU",  "Waist_FU",
    "HDL_FU",    "TGL_FU",  "Serum Cholesterol_FU",
]
for col in numeric_cols:
    if col in df_nd.columns:
        df_nd[col] = pd.to_numeric(df_nd[col], errors="coerce")

# Clean string columns
replace_vals = ["", " ", "NA", "N/A", "na", "n/a", "NIL", "nil",
                "None", "none", "NULL", "null", "-", "--", "nan"]
for col in df_nd.select_dtypes(include="object").columns:
    df_nd[col] = df_nd[col].str.strip()
    df_nd[col] = df_nd[col].replace(replace_vals, np.nan)

# NaN overview 
print(f"\nTotal NaN count: {df_nd.isna().sum().sum()}")
print("\nNaN per column:")
print(df_nd.isna().sum().to_string())

# 1. Kuppuswamy Occupation 
kupp_map = {
    "Politician": 10, "Manager": 10, "Central Government Service": 10,
    "State Government Service": 10, "Tahsildar": 10,
    "Doctor": 9, "Doctor-Dentist": 9, "Doctor-General Physician": 9,
    "Doctor-Ophthalmologist": 9, "Doctor-Homeopathist": 9,
    "Doctor-Pediatrician": 9, "Doctor-Gynecologist": 9,
    "Doctor-Surgeon": 9, "Doctor-Neurologist": 9,
    "Doctor-Siddha": 9, "Doctor-Ayurvedic": 9,
    "Advocate": 9, "Architecture": 9, "AUDITOR": 9,
    "Engineer": 9, "IT Professional": 9,
    "Professor / Teacher / Education": 9, "Doctorate": 9,
    "Reporter": 8, "Accounts/Finance": 8, "Bank": 8,
    "IT Employee": 8, "Supervisor": 8, "Armed Forces": 8, "Police": 8,
    "Clerk": 7, "Government": 7,
    "Business": 6, "Self Employed": 6, "Fashion / Saloon": 6,
    "Retired Employee": 6, "Father In Church": 6, "Priest": 6, "Social Service": 6,
    "Farmer / Agriculture": 5,
    "Private Sector": 4,
    "Driver": 3, "Courier": 3,
    "Daily wages": 2,
    "Housewife": 1, "Retired": 1, "Armed Forces-Retired": 1, "Student": 1,
}

df_nd["Kupp_Occupation"] = df_nd["Occupation"].map(kupp_map)

n_before = len(df_nd)
df_nd = df_nd.dropna(subset=["Kupp_Occupation"])
print(f"\nRemoved (missing/unmapped occupation): {n_before - len(df_nd)} rows")
print(f"Remaining: {len(df_nd)} rows")

# 2. Gender 
# Raw data uses M/F (column is GENDER)
df_nd["Gender"] = df_nd["GENDER"].map({"M": "Male", "F": "Female"})
df_nd["Gender_Code"] = df_nd["Gender"].map({"Male": 1, "Female": 2})

# 3. Age Group 
# 0=<30 | 1=30–39 | 2=40–49 | 3=50–59 | 4=≥60
def age_group(age):
    if pd.isna(age):  return np.nan
    elif age < 30:    return 0
    elif age < 40:    return 1
    elif age < 50:    return 2
    elif age < 60:    return 3
    else:             return 4

df_nd["Age_Group"] = df_nd["AGE"].apply(age_group)
age_labels = {0: "<30", 1: "30–39", 2: "40–49", 3: "50–59", 4: "≥60"}

# Note: No BP columns, AOS, Diab_Duration, or Patient_Status 
# No Engagement_category / Credit_Score / VisitCount in this sheet

# ── Final verification ────────────────────────────────────────────────
print(f"\nAGE — NaN: {df_nd['AGE'].isna().sum()} / {len(df_nd)}")
print(f"\nFinal shape: {df_nd.shape}")
print("\nCoding reference:")
print("  Gender_Code     : 1=Male | 2=Female")
print("  Age_Group       : 0=<30 | 1=30–39 | 2=40–49 | 3=50–59 | 4=≥60")
print("  Kupp_Occupation : 1–10 (Kuppuswamy scale)")
print("  NOTE: No BP columns / AOS / Diab_Duration / Patient_Status in this sheet")

print(df_nd[["MRNO", "Gender", "Gender_Code", "AGE", "Age_Group",
             "Occupation", "Kupp_Occupation"]].head(10).to_string())

In [ ]:
# Step 2: Missingness Check — Non-DiaLA Users; same threshold as before (20%)

outcome_vars = {
    "HbA1c":             ("HbA1c_BL",             "HbA1c_FU"),
    "BMI":               ("BMI_BL",               "BMI_FU"),
    "Waist":             ("Waist_BL",              "Waist_FU"),
    "HDL":               ("HDL_BL",               "HDL_FU"),
    "TGL":               ("TGL_BL",               "TGL_FU"),
    "Serum_Cholesterol": ("Serum Cholesterol_BL",  "Serum Cholesterol_FU"),
}

THRESHOLD = 0.20
n_total   = len(df_nd)

print("=" * 60)
print(f"MISSINGNESS REPORT — NON-DIALA  (n={n_total}, threshold={int(THRESHOLD*100)}%)")
print("=" * 60)
print(f"{'Variable':<22} {'BL Missing':>12} {'FU Missing':>12} {'Include?':>10}")
print("-" * 60)

include_vars = []
exclude_vars = []

for name, (bl_col, fu_col) in outcome_vars.items():
    bl_miss = df_nd[bl_col].isna().sum() / n_total if bl_col in df_nd.columns else 1.0
    fu_miss = df_nd[fu_col].isna().sum() / n_total if fu_col in df_nd.columns else 1.0
    worst   = max(bl_miss, fu_miss)
    flag    = "✓ YES" if worst < THRESHOLD else "✗ EXCLUDE"
    print(f"{name:<22} {bl_miss*100:>10.1f}%  {fu_miss*100:>10.1f}%  {flag:>10}")
    if worst < THRESHOLD:
        include_vars.append(name)
    else:
        exclude_vars.append(name)

print("=" * 60)
print(f"\n✓ INCLUDED ({len(include_vars)}): {include_vars}")
print(f"✗ EXCLUDED ({len(exclude_vars)}): {exclude_vars}")

In [ ]:
# Step 3: Finalise dataframe and calculate deltas for Non-DiaLA users

# Waist excluded (>20% missing per Step 2)
# No AOS, Diab_Duration, Patient_Status in Non-DiaLA sheet

keep_cols = [
    "MRNO", "Gender", "Gender_Code",
    "AGE", "Age_Group",
    "Occupation", "Kupp_Occupation",
    # Clinical outcomes (included from Step 2)
    "HbA1c_BL",             "HbA1c_FU",
    "BMI_BL",               "BMI_FU",
    "HDL_BL",               "HDL_FU",
    "TGL_BL",               "TGL_FU",
    "Serum Cholesterol_BL", "Serum Cholesterol_FU",
]

df_nd = df_nd[[c for c in keep_cols if c in df_nd.columns]].copy()
print(f"Columns kept: {df_nd.shape[1]}")
print(f"Columns: {list(df_nd.columns)}")

# Calculate deltas (FU - BL)
# Negative delta = improvement for all these outcomes
df_nd["Delta_HbA1c"]            = df_nd["HbA1c_FU"]              - df_nd["HbA1c_BL"]
df_nd["Delta_BMI"]               = df_nd["BMI_FU"]               - df_nd["BMI_BL"]
df_nd["Delta_HDL"]               = df_nd["HDL_FU"]               - df_nd["HDL_BL"]
df_nd["Delta_TGL"]               = df_nd["TGL_FU"]               - df_nd["TGL_BL"]
df_nd["Delta_Serum_Cholesterol"] = df_nd["Serum Cholesterol_FU"] - df_nd["Serum Cholesterol_BL"]

# Drop rows missing BL+FU for ALL outcomes 
n_before = len(df_nd)

has_any_complete = (
    (df_nd["HbA1c_BL"].notna()             & df_nd["HbA1c_FU"].notna())             |
    (df_nd["BMI_BL"].notna()               & df_nd["BMI_FU"].notna())               |
    (df_nd["HDL_BL"].notna()               & df_nd["HDL_FU"].notna())               |
    (df_nd["TGL_BL"].notna()               & df_nd["TGL_FU"].notna())               |
    (df_nd["Serum Cholesterol_BL"].notna() & df_nd["Serum Cholesterol_FU"].notna())
)

df_nd = df_nd[has_any_complete].copy()
print(f"\nRows dropped (no complete BL+FU for any outcome): {n_before - len(df_nd)}")
print(f"Final analysis n: {len(df_nd)}")

# SES and Age Group helper columns 
df_nd["SES_Group"] = pd.cut(
    df_nd["Kupp_Occupation"],
    bins=[0, 4, 7, 10],
    labels=["Low (1–4)", "Middle (5–7)", "High (8–10)"]
)
df_nd["SES_Group_str"]  = df_nd["SES_Group"].astype(str)
df_nd["Age_Group_str"]  = df_nd["Age_Group"].map(age_labels)

# Delta summary
delta_cols = [
    "Delta_HbA1c", "Delta_BMI", "Delta_HDL",
    "Delta_TGL",   "Delta_Serum_Cholesterol"
]

print("\n" + "=" * 60)
print("DELTA SUMMARY (FU - BL)")
print("  All vars: negative delta = improvement")
print("  (except HDL: positive delta = improvement)")
print("=" * 60)
print(df_nd[delta_cols].describe().round(3).to_string())

print("\nMissing delta values:")
for col in delta_cols:
    n_miss = df_nd[col].isna().sum()
    print(f"  {col:<28} {n_miss} missing ({n_miss/len(df_nd)*100:.1f}%)")

print(df_nd[["MRNO", "Gender", "Gender_Code", "AGE", "Age_Group",
             "Kupp_Occupation",
             "HbA1c_BL", "HbA1c_FU", "Delta_HbA1c",
             "BMI_BL",   "BMI_FU",   "Delta_BMI"]].head(10).to_string())

In [ ]:
# Step 4: Descriptive Profile — Non-DiaLA Users 

print("=" * 60)
print("DESCRIPTIVE PROFILE — NON-DIALA USERS")
print("=" * 60)

# Continuous variables 
# Note: No AOS or Diab_Duration in Non-DiaLA; this was a recurring issue where a lot of the data wasn't standard across groups and there was quite a bit of missingness.
continuous = {
    "Age (years)":          "AGE",
    "Kuppuswamy SES Score": "Kupp_Occupation",
    "HbA1c BL (%)":        "HbA1c_BL",
    "BMI BL (kg/m²)":      "BMI_BL",
    "HDL BL (mg/dL)":      "HDL_BL",
    "TGL BL (mg/dL)":      "TGL_BL",
    "Serum Chol BL":        "Serum Cholesterol_BL",
}

print(f"\n{'Variable':<30} {'n':>6} {'Mean':>8} {'SD':>8} {'Min':>8} {'Max':>8}")
print("-" * 72)
for label, col in continuous.items():
    if col in df_nd.columns:
        n    = df_nd[col].notna().sum()
        mean = df_nd[col].mean()
        sd   = df_nd[col].std()
        mn   = df_nd[col].min()
        mx   = df_nd[col].max()
        print(f"{label:<30} {n:>6} {mean:>8.2f} {sd:>8.2f} {mn:>8.2f} {mx:>8.2f}")

# Categorical variables
print("\n" + "=" * 60)
print("CATEGORICAL VARIABLES")
print("=" * 60)

# Gender
print("\nGender:")
for val, count in df_nd["Gender"].value_counts(dropna=True).items():
    print(f"  {val:<15} n={count:>4}  ({count/len(df_nd)*100:.1f}%)")

# Age Group
print("\nAge Group:")
for val, count in df_nd["Age_Group"].value_counts(dropna=True).sort_index().items():
    print(f"  {age_labels[val]:<15} n={count:>4}  ({count/len(df_nd)*100:.1f}%)")

# SES — granular
print("\nSES Score — Granular (Kuppuswamy 1–10):")
kupp_labels = {
    1:  "1  — Unskilled/Housewife",
    2:  "2  — Daily wages",
    3:  "3  — Driver/Courier",
    4:  "4  — Private Sector",
    5:  "5  — Farmer/Agriculture",
    6:  "6  — Business/Self-Employed",
    7:  "7  — Clerk/Government",
    8:  "8  — Bank/Supervisor/Armed Forces",
    9:  "9  — Doctor/Engineer/Advocate",
    10: "10 — Politician/Manager/Senior Govt",
}
for val, count in df_nd["Kupp_Occupation"].value_counts(dropna=True).sort_index().items():
    label = kupp_labels.get(int(val), str(val))
    print(f"  {label:<40} n={count:>4}  ({count/len(df_nd)*100:.1f}%)")

# SES — binned
print("\nSES Group — Binned:")
for val, count in df_nd["SES_Group"].value_counts(dropna=True).sort_index().items():
    print(f"  {str(val):<15} n={count:>4}  ({count/len(df_nd)*100:.1f}%)")

print("\n✔ Step 4 complete — paste output and we move to Step 5.")

In [ ]:
# Step 5: Clinical Outcomes BL → FU — Non-DiaLA Users
from scipy import stats
import numpy as np

def cohens_d_paired(a, b):
    diff = a - b
    return diff.mean() / diff.std()

print("=" * 70)
print("STEP 5: CLINICAL OUTCOMES — NON-DIALA USERS (BL → FU)")
print("=" * 70)

outcomes = {
    "HbA1c":            ("HbA1c_BL",             "HbA1c_FU",             "Delta_HbA1c",            "negative"),
    "BMI":              ("BMI_BL",               "BMI_FU",               "Delta_BMI",               "negative"),
    "HDL":              ("HDL_BL",               "HDL_FU",               "Delta_HDL",               "positive"),
    "TGL":              ("TGL_BL",               "TGL_FU",               "Delta_TGL",               "negative"),
    "Serum_Cholesterol":("Serum Cholesterol_BL", "Serum Cholesterol_FU", "Delta_Serum_Cholesterol", "negative"),
}

results_nd = []

for name, (bl_col, fu_col, delta_col, direction) in outcomes.items():
    sub = df_nd[[bl_col, fu_col, delta_col]].dropna()
    n   = len(sub)

    if n < 3:
        print(f"\n{name}: insufficient data (n={n}), skipping.")
        continue

    bl_mean = sub[bl_col].mean()
    bl_sd   = sub[bl_col].std()
    fu_mean = sub[fu_col].mean()
    fu_sd   = sub[fu_col].std()
    delta   = sub[delta_col].mean()

    # Normality check
    delta_vals = sub[delta_col]
    if len(delta_vals) > 5000:
        delta_sample = delta_vals.sample(5000, random_state=42)
    else:
        delta_sample = delta_vals

    shapiro_stat, shapiro_p = stats.shapiro(delta_sample)
    normal = shapiro_p > 0.05

    # Paired test 
    if normal:
        test_name = "Paired t-test"
        t_stat, p_val = stats.ttest_rel(sub[bl_col], sub[fu_col])
    else:
        test_name = "Wilcoxon"
        t_stat, p_val = stats.wilcoxon(sub[bl_col], sub[fu_col])

    cd  = cohens_d_paired(sub[bl_col], sub[fu_col])
    sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"

    if direction == "negative":
        improved = "✓ improved" if delta < 0 else "✗ worsened"
    else:
        improved = "✓ improved" if delta > 0 else "✗ worsened"

    results_nd.append({
        "Variable":   name,
        "n":          n,
        "BL Mean±SD": f"{bl_mean:.2f} ± {bl_sd:.2f}",
        "FU Mean±SD": f"{fu_mean:.2f} ± {fu_sd:.2f}",
        "Delta":      round(delta, 3),
        "Test":       test_name,
        "Stat":       round(t_stat, 3),
        "p_value":    round(p_val, 4),
        "Sig":        sig,
        "Cohens_D":   round(abs(cd), 3),
        "Direction":  improved,
    })

    print(f"\n{'─' * 70}")
    print(f"{name} {sig}  |  {improved}  |  n={n}")
    print(f"  BL:        {bl_mean:.2f} ± {bl_sd:.2f}")
    print(f"  FU:        {fu_mean:.2f} ± {fu_sd:.2f}")
    print(f"  Delta:     {delta:.3f}")
    print(f"  Test:      {test_name} | stat={t_stat:.3f} | p={p_val:.4f} {sig}")
    print(f"  Cohen's D: {abs(cd):.3f}")
    print(f"  Normality: SW p={shapiro_p:.3f} ({'normal' if normal else 'non-normal'})")

# Summary table 
print(f"\n{'=' * 70}")
print("SUMMARY TABLE — NON-DIALA USERS")
print(f"{'=' * 70}")
print(f"{'Variable':<22} {'n':>5} {'BL Mean±SD':<18} {'FU Mean±SD':<18} "
      f"{'Delta':>8} {'p-value':>8} {'Sig':>5} {'CohenD':>8} {'Result'}")
print("-" * 105)
for r in results_nd:
    print(f"{r['Variable']:<22} {r['n']:>5} {r['BL Mean±SD']:<18} {r['FU Mean±SD']:<18} "
          f"{r['Delta']:>8} {r['p_value']:>8} {r['Sig']:>5} {r['Cohens_D']:>8} {r['Direction']}")

In [ ]:
# Step 6: Does SES predict clinical improvement? for Non-DiaLA users 
from scipy import stats
from itertools import combinations
import matplotlib.pyplot as plt
import numpy as np

print("=" * 70)
print("STEP 6: SES vs CLINICAL IMPROVEMENT — NON-DIALA USERS")
print("=" * 70)

delta_outcomes = {
    "HbA1c":            ("Delta_HbA1c",            "negative"),
    "BMI":              ("Delta_BMI",               "negative"),
    "HDL":              ("Delta_HDL",               "positive"),
    "TGL":              ("Delta_TGL",               "negative"),
    "Serum_Cholesterol":("Delta_Serum_Cholesterol", "negative"),
}

ses_order = ["Low (1–4)", "Middle (5–7)", "High (8–10)"]

# Manual Dunn's post-hoc
def dunn_posthoc(data, group_col, val_col, groups):
    from scipy.stats import rankdata
    all_vals   = data[val_col].values
    all_groups = data[group_col].values
    n_total    = len(all_vals)
    ranks      = rankdata(all_vals)
    pairs      = list(combinations(groups, 2))
    results    = {}
    for g1, g2 in pairs:
        idx1 = all_groups == g1
        idx2 = all_groups == g2
        n1, n2 = idx1.sum(), idx2.sum()
        r1, r2 = ranks[idx1].mean(), ranks[idx2].mean()
        se = np.sqrt((n_total * (n_total + 1) / 12.0) * (1.0 / n1 + 1.0 / n2))
        z  = (r1 - r2) / se
        p  = 2 * stats.norm.sf(abs(z))
        results[(g1, g2)] = p
    n_pairs   = len(pairs)
    corrected = {k: min(v * n_pairs, 1.0) for k, v in results.items()}
    return corrected

step6_results_nd = []

for name, (delta_col, direction) in delta_outcomes.items():

    sub = df_nd[["SES_Group_str", "Kupp_Occupation", delta_col]].dropna().copy()
    n   = len(sub)

    print(f"\n{'─' * 70}")
    print(f"{name}  |  n={n}")

    # Group means 
    print(f"\n  Group means:")
    for grp in ses_order:
        grp_data = sub[sub["SES_Group_str"] == grp][delta_col]
        if len(grp_data) > 0:
            print(f"    {grp:<18} n={len(grp_data):>4}  "
                  f"mean={grp_data.mean():>7.3f}  sd={grp_data.std():>7.3f}")

    # ruskal-Wallis
    groups = [
        sub[sub["SES_Group_str"] == g][delta_col].dropna().values
        for g in ses_order
    ]
    groups = [g for g in groups if len(g) >= 3]

    if len(groups) < 2:
        print("  Insufficient groups for Kruskal-Wallis, skipping.")
        continue

    kw_stat, kw_p = stats.kruskal(*groups)
    kw_sig = "***" if kw_p < 0.001 else "**" if kw_p < 0.01 else "*" if kw_p < 0.05 else "ns"
    print(f"\n  Kruskal-Wallis: H={kw_stat:.3f}, p={kw_p:.4f} {kw_sig}")

    # Dunn's post-hoc (if sig.)
    if kw_p < 0.05:
        print("  Dunn's post-hoc (Bonferroni corrected):")
        dunn_results = dunn_posthoc(
            sub, group_col="SES_Group_str", val_col=delta_col, groups=ses_order
        )
        for (g1, g2), p in dunn_results.items():
            sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
            print(f"    {str(g1):<18} vs {str(g2):<18} p={p:.4f} {sig}")
    else:
        print("  No significant difference — post-hoc not run.")

    # Spearman 
    spear_r, spear_p = stats.spearmanr(sub["Kupp_Occupation"], sub[delta_col])
    spear_sig = "***" if spear_p < 0.001 else "**" if spear_p < 0.01 else "*" if spear_p < 0.05 else "ns"
    print(f"\n  Spearman (Kupp score vs delta): r={spear_r:.3f}, "
          f"p={spear_p:.4f} {spear_sig}")

    step6_results_nd.append({
        "Variable":     name,
        "n":            n,
        "KW_H":         round(kw_stat, 3),
        "KW_p":         round(kw_p, 4),
        "KW_sig":       kw_sig,
        "Spearman_r":   round(spear_r, 3),
        "Spearman_p":   round(spear_p, 4),
        "Spearman_sig": spear_sig,
    })

# Summary table 
print(f"\n{'=' * 70}")
print("STEP 6 SUMMARY TABLE — NON-DIALA USERS")
print(f"{'=' * 70}")
print(f"{'Variable':<22} {'n':>5} {'KW H':>8} {'KW p':>8} {'Sig':>5} "
      f"{'Spearman r':>12} {'Spearman p':>12} {'Sig':>5}")
print("-" * 80)
for r in step6_results_nd:
    print(f"{r['Variable']:<22} {r['n']:>5} {r['KW_H']:>8} {r['KW_p']:>8} "
          f"{r['KW_sig']:>5} {r['Spearman_r']:>12} {r['Spearman_p']:>12} "
          f"{r['Spearman_sig']:>5}")

# Box plots 
colors = ["#d9534f", "#f0ad4e", "#5cb85c"]

n_outcomes = len(delta_outcomes)
fig, axes = plt.subplots(1, n_outcomes, figsize=(5 * n_outcomes, 5))
if n_outcomes == 1:
    axes = [axes]

for i, (name, (delta_col, direction)) in enumerate(delta_outcomes.items()):
    sub = df_nd[["SES_Group_str", delta_col]].dropna().copy()
    data_by_group = [
        sub[sub["SES_Group_str"] == g][delta_col].values
        for g in ses_order
    ]
    ax = axes[i]
    bp = ax.boxplot(data_by_group, tick_labels=ses_order, patch_artist=True)
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.axhline(0, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
    ax.set_title(f"{name} — Delta by SES Group", fontweight="bold")
    ax.set_xlabel("SES Group")
    ax.set_ylabel(f"Delta {name} (FU - BL)")
    ax.tick_params(axis="x", rotation=15)

plt.suptitle("Clinical Improvement by SES Group — Non-DiaLA Users",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("step6_nondiala_ses_vs_deltas.png", dpi=150, bbox_inches="tight")
plt.close()